### Notebook for selecting LENS examples for Qualitative Analysis + calculating averages of __

In [1]:
import subprocess, pandas as pd, numpy as np

exp_csv    = "csv/M-7.csv"
lens_script = "lens_score_OR.py"
lens_venv = "/home/c23068554/miniconda3/envs/lens_eval/bin/python"
#sentences per range
N = 10


## Sampling sentences

In [2]:
# Getting individual LENS
lens_venv = "/home/c23068554/miniconda3/envs/lens_eval/bin/python"
def get_lens_scores(csv_path):
    result = subprocess.run([lens_venv, "lens_score_OR.py", csv_path, "--per-row"], capture_output=True, text=True)
    lines = [l for l in result.stdout.strip().split("\n") if l.replace(".", "").replace("-", "").isdigit()]
    return list(map(float, lines))

df = pd.read_csv(exp_csv)
df["LENS"] = get_lens_scores(exp_csv)
df.head(3)

,Exp ID,Index,Original,Reference,LM Output,LENS
0,M-7,0,The exhibitions served a number of purposes - ...,The exhibitions served a number of purposes. T...,The exhibitions had several goals. Their main ...,74.080138
1,M-7,1,To aid biodiversity conservation we have drawn...,To aid biodiversity conservation we have drawn...,"To aid biodiversity conservation, we have crea...",58.121420
2,M-7,2,"If notice of the sale hasn ’ t been given, the...","If notice of the sale hasn’t been given, the r...","If notice of the sale hasn't been given, the t...",58.602677


### Divide scores to percentiles and sample

In [3]:
rState = 42
N = 10

p90 = np.percentile(df.LENS, 90)
p55 = np.percentile(df.LENS, 55)
p45 = np.percentile(df.LENS, 45)
p10 = np.percentile(df.LENS, 10)

top = df[df.LENS >= p90].sample(N, random_state= rState)
median = df[df.LENS.between(p45, p55)].sample(N, random_state=rState)
bottom = df[df.LENS <= p10].sample(N, random_state=rState)

In [4]:
top["percentile"]    = "top"
median["percentile"] = "median"
bottom["percentile"] = "bottom"

In [5]:
sample = pd.concat([top, median, bottom])[["percentile", "Index", "LENS", "Original", "Reference", "LM Output"]]
sample.to_csv("QA_examples.csv", index=False)

## Calculating word count averages

In [6]:
# Average word count per percentile — Original vs LM Output
for percentile, sentences in sample.groupby("percentile"):
    orig = sentences["Original"].str.split().str.len().mean()
    out  = sentences["LM Output"].str.split().str.len().mean()
    print(f"{percentile}: original={orig:.1f}  output={out:.1f}  diff={out-orig:+.1f}")

# Average word count of the whole sample
totalOrig = sample["Original"].str.split().str.len().mean()
totalOut  = sample["LM Output"].str.split().str.len().mean()
print(f"all: original={totalOrig:.1f} output={totalOut:.1f}")

bottom: original=26.2  output=23.3  diff=-2.9
median: original=28.7  output=24.2  diff=-4.5
top: original=29.5  output=24.5  diff=-5.0
all: original=28.1 output=24.0
